# 01 - Data Understanding

**Phishing Website Detection - Exploratory Data Understanding**

This notebook loads the UCI Phishing Websites dataset, inspects its shape,
schema, class balance, and per-feature distributions. All operations go
through the project's modular `src.data.DataIngestion` layer so the same
code path is used in training, batch inference, and the Streamlit app.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import get_config, setup_logging, get_logger
from src.data import DataIngestion, DataValidation

setup_logging()
cfg = get_config()
log = get_logger("notebook_01")


## 1. Load the raw dataset

In [ ]:
ing = DataIngestion(cfg.raw_data_dir,
            train_file=cfg.config.files.train_data,
            test_file=cfg.config.files.test_data)
art = ing.ingest()
train_df = art.train_df
test_df = art.test_df
print(f"Train shape: {train_df.shape}")
print(f"Test  shape: {test_df.shape}")
train_df.head()


## 2. Schema validation
Enforces every feature ∈ {-1, 0, 1} and Result ∈ {-1, 1}.

In [ ]:
validator = DataValidation(
    schema_cfg=cfg.schema,
    features=cfg.features,
    target_name=cfg.target_name,
)
report = validator.validate(train_df, test_df, cfg.validated_data_dir)
print(f"Valid: {report.is_valid}")
print(f"Class balance: {pd.Series(train_df[cfg.target_name]).value_counts().to_dict()}")


## 3. Per-feature value distribution

In [ ]:
feature_value_counts = pd.DataFrame({
    f: train_df[f].value_counts().sort_index().to_dict()
    for f in cfg.features
}).T.fillna(0).astype(int)
feature_value_counts.head(15)


## 4. Class balance

In [ ]:
ax = train_df['Result'].value_counts().plot(
    kind='bar', color=['#FF3B3B', '#00E5A8'],
    figsize=(6,4), title='Class balance (Result)')
ax.set_xticklabels(['Phishing (-1)', 'Legitimate (+1)'], rotation=0)
ax.set_ylabel('Count')
plt.tight_layout(); plt.show()


## 5. Per-feature class-conditional distribution

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(20, 18))
for ax, feat in zip(axes.ravel(), cfg.features):
    ct = pd.crosstab(train_df[feat], train_df['Result'])
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=['#FF3B3B', '#00E5A8'], legend=False)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
plt.tight_layout(); plt.show()


## 6. Missing values

In [ ]:
print(f"Train missing: {train_df.isna().sum().sum()}")
print(f"Test  missing: {test_df.isna().sum().sum()}")


## Summary

The dataset has **11,055 rows × 31 columns**, **no missing values**, and a
class balance of **~44% phishing / ~56% legitimate** - matching the published
UCI Phishing Websites specification. All 30 features are ternary-encoded
(-1, 0, 1), making schema validation a one-liner. The data is ready for
feature engineering and model training.
